# 3施設グリオーマ IDH コホート — データ分割の定義

**このノートブックは、全実験で用いる train/val/test 分割の生成方法を記録する「唯一の正典（single source of truth）」です。**
上から順に再実行すると、`data/preprocessed/v1.0.0/_global/splits/` 内の CSV を完全に同一に再生成します
（`seed=42` と入力 metadata が同じであれば決定的に一致）。

- **論文用の成果物**: Methods セクションでは本ノートブック（および凍結された CSV）を引用してください。
- **入力**（metadata のみ・画像処理は不要）:
  - `data/metadata/UCSF-PDGM-metadata_v5.csv`
  - `data/metadata/UTSW_Glioma_Metadata.tsv`
  - `data/metadata/UPENN-GBM_clinical_info_v2.1.csv`, `UPENN-GBM_acquisition.csv`
  - `data/preprocessed/v1.0.0/utsw-glioma/_global/seg_source_inventory.csv`（UTSW の QC ゲート）
- **出力**: `splits_loso_foldA.csv`, `splits_loso_foldB.csv`, `splits_vendor_philips.csv`,
  `splits_field.csv`, `splits_random_5fold.csv`（＋ドキュメントとしての本ノートブック）

## 1. 分割設計の意図

研究の目的は MRI からの **IDH 変異状態** 予測です。本分割は、実運用の現実である
**施設間（サイト間）の分布外汎化（out-of-distribution generalization）** を測定できるよう設計しています。

**主評価 — サイト間 leave-one-site-out（LOSO, 2 fold）**
- Fold A: test = **UCSF**, train = UTSW + UPenn
- Fold B: test = **UTSW**, train = UCSF + UPenn
- **UPenn は決して test 施設にしない。** IDH-Mut がわずか 16 例しかなく、hold-out 施設にすると
  陽性クラスの指標（Mut に対する AUC / recall）が極端に不安定になるため、常に train 側に置く。

**補助評価1 — UTSW 内 leave-one-vendor-out**
UTSW は唯一のマルチベンダー施設。Siemens+GE で学習し（少数ベンダーの Hitachi/Toshiba/Not-Reported も train 側）、
**Philips** で検証。これにより「施設・母集団のシフト」と「スキャナーメーカーのシフト」を分離する。

**補助評価2 — UTSW 内 磁場強度シフト**
**3T** で検証、残り（1.5T ＋ 少数/Not-Reported の磁場）で学習。磁場強度のドメインシフトを分離する。

**ベースライン — 全施設混合の層化ランダム 5-fold**
in-distribution（分布内）の楽観的な基準。これと LOSO の差がドメインシフトによる性能低下量を定量化する。

**すべての分割で守る不変条件:**
1. **患者単位グループ化** — 同一患者が train と test に跨がらない。UCSF には baseline＋follow-up の
   2 セッションを持つ患者が 6 名いるが、両セッションは必ず同じ側に置く。
2. **IDH 層化** — 可能な限り Mut/WT 比率を各 role で維持する。
3. **ランダム分割の施設比率** — 層化キーを `site × IDH` にすることで、施設構成とクラス比率の両方を各 fold で保つ。
4. **IDH-NA は完全除外** — UTSW の NA 3 例、UPenn の `NOS/NEC`（判定不能）96 例は分割前に除外する。

**内側 validation:** 各外側 `train` をさらに 80/20 で inner-train / `val` に分割
（患者単位グループ化＋`site×IDH` 層化）。ハイパラ選択と early stopping は `val` を用い、外側 `test` は一度だけ評価する。
**`seed = 42`** を全所に適用。

In [1]:
import pandas as pd, numpy as np, re, os
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42

# ノートブックの位置からリポジトリのルートを解決する（リポを移動しても動くように）。
root = Path.cwd()
for _ in range(8):
    if (root / "data" / "metadata").exists():
        break
    root = root.parent
assert (root / "data" / "metadata").exists(), "data/metadata を含むリポジトリのルートが見つかりません"

MD   = root / "data" / "metadata"

PROC = root / "data" / "preprocessed" / "v1.0.0"
OUT  = PROC / "_global" / "splits"
OUT.mkdir(parents=True, exist_ok=True)
print("リポジトリルート :", root)
print("出力先 -> :", OUT)

リポジトリルート : /home/llmteam0203/Scripts/python/OpenIDH
出力先 -> : /home/llmteam0203/Scripts/python/OpenIDH/data/preprocessed/v1.0.0/_global/splits


## 2. コホートの読み込みと施設ごとの IDH 変換ルール

施設ごとに IDH の表記が異なるため、変換は施設別に明示する:

| 施設 | IDH 列 | → WT | → Mut | → 除外（NA） |
|------|-------|------|-------|--------------|
| UCSF | `IDH` | `wildtype` | 分子レベルの記載すべて（`IDH1 p.R132H`, `mutated (NOS)` …） | なし |
| UTSW | `IDH` | `wild type` | `mutated` | 空欄/NA 3 例 |
| UPenn| `IDH1`| `Wildtype` | `Mutated` | `NOS/NEC`（判定不能）96 例 |

注意: **UCSF** の `mutated (NOS)` は「変異は確定・サブタイプ不明」→ **Mut**、一方 **UPenn** の `NOS/NEC` は
「IDH を分類できない」→ **NA**。意味が正反対なので別々に扱う。

In [2]:
def _num(x):
    m = re.search(r"(\d+)", str(x));  return int(m.group(1)) if m else None

# ---- UCSF: metadata 全 501 セッション。患者キーは数値 ID ----
u = pd.read_csv(MD / "UCSF-PDGM-metadata_v5.csv")
def ucsf_idh(x):
    s = str(x).strip().lower()
    return "WT" if s in ("wildtype", "wild type", "wt") else "Mut"  # 残りはすべて真の変異
ucsf = pd.DataFrame({
    "subject_id": u["ID"].astype(str),
    "site": "UCSF",
    "idh":  u["IDH"].map(ucsf_idh),
    "patient_id": "UCSF-" + u["ID"].map(_num).astype(str),
    "vendor": "NA", "field": "NA",
})

# ---- UTSW: qc_passed==true(622) の後、IDH-NA を除外 -> 619 ----
inv = pd.read_csv(PROC / "utsw-glioma" / "_global" / "seg_source_inventory.csv")
inv = inv[inv["qc_passed"].astype(str).str.lower().isin(["true", "1"])]
qc_ids = set(inv["subject_id"])
t = pd.read_csv(MD / "UTSW_Glioma_Metadata.tsv", sep="\t")
t = t[t["Subject ID"].isin(qc_ids)].copy()
def utsw_idh(x):
    s = str(x).strip().lower()
    if "wild" in s: return "WT"
    if "mut"  in s: return "Mut"
    return "NA"
t["idh"] = t["IDH"].map(utsw_idh)
t = t[t["idh"] != "NA"].copy()
utsw = pd.DataFrame({
    "subject_id": t["Subject ID"].astype(str),
    "site": "UTSW",
    "idh":  t["idh"].values,
    "patient_id": "UTSW-" + t["Subject ID"].astype(str),
    "vendor": t["Scanner Make"].astype(str).str.strip().values,
    "field":  t["Scanner Strength"].astype(str).str.strip().values,
})

# ---- UPenn: clinical ∩ 前処理済み画像(611) の後、NOS/NEC を除外 -> 515 ----
cl  = pd.read_csv(MD / "UPENN-GBM_clinical_info_v2.1.csv")
acq = pd.read_csv(MD / "UPENN-GBM_acquisition.csv")[["ID", "Manufacturer", "Magnetic Field Strength"]]
imaged = {d for d in os.listdir(PROC / "upenn-gbm")
          if (PROC / "upenn-gbm" / d).is_dir() and d.startswith("UPENN")}
cl = cl[cl["ID"].isin(imaged)].copy()
def upenn_idh(x):
    s = str(x).strip().lower()
    if s == "wildtype": return "WT"
    if s == "mutated":  return "Mut"
    return "NA"          # NOS/NEC など
cl["idh"] = cl["IDH1"].map(upenn_idh)
cl = cl[cl["idh"] != "NA"].merge(acq, on="ID", how="left")
upenn = pd.DataFrame({
    "subject_id": cl["ID"].astype(str),
    "site": "UPenn",
    "idh":  cl["idh"].values,
    "patient_id": "UPenn-" + cl["ID"].astype(str).str.replace(r"_\d+$", "", regex=True),
    "vendor": cl["Manufacturer"].astype(str).str.strip().values,
    "field":  cl["Magnetic Field Strength"].astype(str).str.strip().values,
})

df = pd.concat([ucsf, utsw, upenn], ignore_index=True)
df["y"]     = (df["idh"] == "Mut").astype(int)
df["strat"] = df["site"] + "_" + df["idh"]
len(df)

1635

## 3. コホート数の検算と患者グループ化の確認

凍結したコホート数を確認し、患者単位グループ化が正しいこと（マルチセッション患者は UCSF のみ）を検証する。

In [3]:
def cc(d): return dict(n=len(d), Mut=int((d.idh=="Mut").sum()), WT=int((d.idh=="WT").sum()))
print("コホート数:")
for s in ["UCSF","UTSW","UPenn"]:
    print(f"  {s:6s}", cc(df[df.site==s]))
print("  合計  ", cc(df), f"陽性率={df.y.mean():.3f}")

print("\n患者グループ化:")
for s in ["UCSF","UTSW","UPenn"]:
    d = df[df.site==s]; vc = d.patient_id.value_counts(); multi = vc[vc>1]
    print(f"  {s:6s} 患者数={d.patient_id.nunique()} セッション数={len(d)} マルチセッション={multi.to_dict()}")

assert cc(df) == dict(n=1635, Mut=295, WT=1340), "コホート数が凍結定義からずれています！"

コホート数:
  UCSF   {'n': 501, 'Mut': 103, 'WT': 398}
  UTSW   {'n': 619, 'Mut': 176, 'WT': 443}
  UPenn  {'n': 515, 'Mut': 16, 'WT': 499}
  合計   {'n': 1635, 'Mut': 295, 'WT': 1340} 陽性率=0.180

患者グループ化:
  UCSF   患者数=495 セッション数=501 マルチセッション={'UCSF-429': 2, 'UCSF-396': 2, 'UCSF-409': 2, 'UCSF-431': 2, 'UCSF-391': 2, 'UCSF-433': 2}
  UTSW   患者数=619 セッション数=619 マルチセッション={}
  UPenn  患者数=515 セッション数=515 マルチセッション={}


## 4. 分割ヘルパー — 内側 80/20（患者単位グループ化・層化）

`StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` の最初の fold（約20%）を `val` にする。
これにより inner-train/val の境界を患者が跨がず、`site×IDH` のバランスも保たれる。

In [4]:
def inner_split(train_df, seed=SEED):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for tr, va in sgkf.split(train_df, train_df["strat"], train_df["patient_id"]):
        return train_df.index[tr], train_df.index[va]   # 最初の fold を約20%の val に

def assign_roles(train_df, test_df, seed=SEED):
    roles = pd.Series("train", index=pd.concat([train_df, test_df]).index)
    _, iva = inner_split(train_df, seed)
    roles.loc[iva] = "val"
    roles.loc[test_df.index] = "test"
    return roles

def write_split(roles, fold_id, fname):
    out = df.loc[roles.index, ["subject_id","site","idh"]].copy()
    out["split_role"] = roles.values
    out["fold_id"]    = fold_id
    out.to_csv(OUT / fname, index=False)
    return out

def summarize(roles, label):
    rows=[]
    for r in ["train","val","test"]:
        sub = df.loc[roles[roles==r].index]
        if len(sub)==0: continue
        mut=int((sub.idh=="Mut").sum()); wt=int((sub.idh=="WT").sum())
        rows.append(dict(split=label, role=r, n=len(sub), Mut=mut, WT=wt,
                         pos=f"{mut/len(sub)*100:.1f}%"))
    return rows

summary = []

## 5. 主評価 — サイト間 LOSO（Fold A: test=UCSF, Fold B: test=UTSW）
どちらの fold でも UPenn は train に留まる。

In [5]:
for fold, test_site in [("A","UCSF"), ("B","UTSW")]:
    test_df  = df[df.site == test_site]
    train_df = df[df.site != test_site]
    roles = assign_roles(train_df, test_df)
    write_split(roles, f"loso_{fold}", f"splits_loso_fold{fold}.csv")
    summary += summarize(roles, f"LOSO Fold {fold} (test={test_site})")
pd.DataFrame([r for r in summary if r["split"].startswith("LOSO")])

,split,role,n,Mut,WT,pos
0,LOSO Fold A (test=UCSF),train,907,154,753,17.0%
1,LOSO Fold A (test=UCSF),val,227,38,189,16.7%
2,LOSO Fold A (test=UCSF),test,501,103,398,20.6%
3,LOSO Fold B (test=UTSW),train,812,95,717,11.7%
4,LOSO Fold B (test=UTSW),val,204,24,180,11.8%
5,LOSO Fold B (test=UTSW),test,619,176,443,28.4%


## 6. 補助評価1 — UTSW 内 leave-one-vendor-out（test = Philips）
少数ベンダー（Hitachi/Toshiba/Not-Reported）は train 側に置く。

In [6]:
utsw_df = df[df.site == "UTSW"]
print("UTSW ベンダー内訳:", utsw_df.vendor.value_c
    ounts().to_dict())
philips = utsw_df[utsw_df.vendor.str.contains("Philips", case=False, na=False)]
train_v = utsw_df.drop(philips.index)
roles_v = assign_roles(train_v, philips)
write_split(roles_v, "vendor_philips", "splits_vendor_philips.csv")
summary += summarize(roles_v, "Vendor (UTSW: test=Philips)")
pd.DataFrame([r for r in summary if r["split"].startswith("Vendor")])

UTSW ベンダー内訳: {'Siemens': 265, 'GE': 170, 'Philips': 140, 'Not Reported': 28, 'Hitachi': 15, 'Toshiba': 1}


,split,role,n,Mut,WT,pos
0,Vendor (UTSW: test=Philips),train,383,102,281,26.6%
1,Vendor (UTSW: test=Philips),val,96,25,71,26.0%
2,Vendor (UTSW: test=Philips),test,140,49,91,35.0%


## 7. 補助評価2 — UTSW 内 磁場強度（test = 3T, train = 残り）
少数/Not-Reported の磁場強度は train 側に置く。

In [7]:
def is3T(x):
    try: return abs(float(x) - 3.0) < 1e-6
    except: return False
print("UTSW 磁場内訳:", utsw_df.field.value_counts().to_dict())
three   = utsw_df[utsw_df.field.map(is3T)]
train_f = utsw_df.drop(three.index)
roles_f = assign_roles(train_f, three)
write_split(roles_f, "field_3T", "splits_field.csv")
summary += summarize(roles_f, "Field (UTSW: test=3T, train=rest)")
pd.DataFrame([r for r in summary if r["split"].startswith("Field")])

UTSW 磁場内訳: {'1.5': 379, '3': 195, 'Not Reported': 28, '0.7': 6, '1.16': 5, '1': 3, '0.3': 2, '0.94999999': 1}


,split,role,n,Mut,WT,pos
0,"Field (UTSW: test=3T, train=rest)",train,339,99,240,29.2%
1,"Field (UTSW: test=3T, train=rest)",val,85,24,61,28.2%
2,"Field (UTSW: test=3T, train=rest)",test,195,53,142,27.2%


## 8. ベースライン — 全施設 層化グループ ランダム 5-fold
`site×IDH` で層化し、患者でグループ化。各被験者はちょうど 1 つの fold で `test` になる。

In [8]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
rand_out = []
for k, (tr, te) in enumerate(sgkf.split(df, df["strat"], df["patient_id"])):
    roles = assign_roles(df.iloc[tr], df.iloc[te])
    o = df.loc[roles.index, ["subject_id","site","idh"]].copy()
    o["split_role"] = roles.values; o["fold_id"] = k
    rand_out.append(o)
    summary += summarize(roles, f"Random 5-fold [fold {k}]")
pd.concat(rand_out).to_csv(OUT / "splits_random_5fold.csv", index=False)
pd.DataFrame([r for r in summary if r["split"].startswith("Random")])

,split,role,n,Mut,WT,pos
0,Random 5-fold [fold 0],train,1046,188,858,18.0%
1,Random 5-fold [fold 0],val,262,48,214,18.3%
2,Random 5-fold [fold 0],test,327,59,268,18.0%
3,Random 5-fold [fold 1],train,1046,189,857,18.1%
4,Random 5-fold [fold 1],val,262,47,215,17.9%
5,Random 5-fold [fold 1],test,327,59,268,18.0%
6,Random 5-fold [fold 2],train,1046,189,857,18.1%
7,Random 5-fold [fold 2],val,262,47,215,17.9%
8,Random 5-fold [fold 2],test,327,59,268,18.0%
9,Random 5-fold [fold 3],train,1046,189,857,18.1%


## 9. 全体の症例数テーブル（train = inner-train, val = inner-val, test）

In [9]:
pd.DataFrame(summary)[["split","role","n","Mut","WT","pos"]]

,split,role,n,Mut,WT,pos
0,LOSO Fold A (test=UCSF),train,907,154,753,17.0%
1,LOSO Fold A (test=UCSF),val,227,38,189,16.7%
2,LOSO Fold A (test=UCSF),test,501,103,398,20.6%
3,LOSO Fold B (test=UTSW),train,812,95,717,11.7%
4,LOSO Fold B (test=UTSW),val,204,24,180,11.8%
5,LOSO Fold B (test=UTSW),test,619,176,443,28.4%
6,Vendor (UTSW: test=Philips),train,383,102,281,26.6%
7,Vendor (UTSW: test=Philips),val,96,25,71,26.0%
8,Vendor (UTSW: test=Philips),test,140,49,91,35.0%
9,"Field (UTSW: test=3T, train=rest)",train,339,99,240,29.2%


## 10. リーク / 整合性チェック（assert 済み — 全て通過する必要あり）
すべての分割・fold について: (1) train/val と test で `subject_id` の重複がないこと、
(2) 患者が train/val 対 test の境界を跨がないこと、(3) IDH-NA が混入していないこと。

In [10]:
files = {
 "loso_foldA":"splits_loso_foldA.csv", "loso_foldB":"splits_loso_foldB.csv",
 "vendor_philips":"splits_vendor_philips.csv", "field":"splits_field.csv",
 "random_5fold":"splits_random_5fold.csv",
}
sub2pat = dict(zip(df.subject_id, df.patient_id))
known   = set(df.subject_id)
all_ok  = True
for name, fname in files.items():
    s = pd.read_csv(OUT / fname)
    for fid in s.fold_id.unique():
        sf = s[s.fold_id == fid]
        tr = set(sf[sf.split_role=="train"].subject_id)
        va = set(sf[sf.split_role=="val"].subject_id)
        te = set(sf[sf.split_role=="test"].subject_id)
        overlap   = (tr & te) | (va & te) | (tr & va)
        pat_leak  = {sub2pat[i] for i in tr|va} & {sub2pat[i] for i in te}
        na        = [i for i in sf.subject_id if i not in known] + list(sf[sf.idh=="NA"].subject_id)
        ok = not overlap and not pat_leak and not na
        all_ok &= ok
        print(f"  {name:15s} fold={str(fid):14s} 重複={len(overlap)} 患者リーク={len(pat_leak)} NA={len(na)} -> {'OK' if ok else 'FAIL'}")
assert all_ok, "整合性チェックに失敗しました"
print("\n全チェック通過 — 分割を書き出しました:", OUT)

  loso_foldA      fold=loso_A         重複=0 患者リーク=0 NA=0 -> OK
  loso_foldB      fold=loso_B         重複=0 患者リーク=0 NA=0 -> OK
  vendor_philips  fold=vendor_philips 重複=0 患者リーク=0 NA=0 -> OK
  field           fold=field_3T       重複=0 患者リーク=0 NA=0 -> OK
  random_5fold    fold=0              重複=0 患者リーク=0 NA=0 -> OK
  random_5fold    fold=1              重複=0 患者リーク=0 NA=0 -> OK
  random_5fold    fold=2              重複=0 患者リーク=0 NA=0 -> OK
  random_5fold    fold=3              重複=0 患者リーク=0 NA=0 -> OK
  random_5fold    fold=4              重複=0 患者リーク=0 NA=0 -> OK

全チェック通過 — 分割を書き出しました: /home/llmteam0203/Scripts/python/OpenIDH/data/preprocessed/v1.0.0/_global/splits
